In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel('../data/Financial_analysis_Data_Set.xlsx')
print(df.shape)
print(df.head())

(24, 8)
       Month  Development Costs  Operatinal Costs  Marketing Costs  \
0 2023-01-01             107000             12000            10000   
1 2023-02-01             107000             12000             5000   
2 2023-03-01             108000             12000             5000   
3 2023-04-01             107000             12000            10000   
4 2023-05-01             111100             12000             4000   

   Travelling Cost  Training Cost  Maintenance Cost Expenses  
0             4000           6000              4000   Budget  
1             4000           6000              4000   Budget  
2             4000           6000              4000   Budget  
3             4000           6000              4000   Budget  
4             4000           6000              4000   Budget  


In [5]:
budget = df[df['Expenses'] == 'Budget'].copy().reset_index(drop=True)
actual = df[df['Expenses'] == 'Actual'].copy().reset_index(drop=True)

cost_cols = ['Development Costs', 'Operatinal Costs', 'Marketing Costs',
             'Travelling Cost', 'Training Cost', 'Maintenance Cost']

budget['Total'] = budget[cost_cols].sum(axis=1)
actual['Total'] = actual[cost_cols].sum(axis=1)

variance = actual[cost_cols].values - budget[cost_cols].values
variance_df = pd.DataFrame(variance, columns=cost_cols)
variance_df['Month'] = budget['Month']
variance_df['Total_Variance'] = actual['Total'].values - budget['Total'].values

print("=== BUDGET vs ACTUAL VARIANCE ===")
print(variance_df[['Month', 'Total_Variance']].to_string())
print(f"\nAvg Monthly Variance: {variance_df['Total_Variance'].mean():,.0f}")
print(f"Most Over-Budget Category: {variance_df[cost_cols].mean().idxmax()}")
print(f"Most Under-Budget Category: {variance_df[cost_cols].mean().idxmin()}")

=== BUDGET vs ACTUAL VARIANCE ===
        Month  Total_Variance
0  2023-01-01            1270
1  2023-02-01           -2146
2  2023-03-01            2853
3  2023-04-01            5990
4  2023-05-01           -3132
5  2023-06-01             860
6  2023-07-01          -15267
7  2023-08-01          -10200
8  2023-09-01          -11930
9  2023-10-01           -8762
10 2023-11-01             154
11 2023-12-01           -3710

Avg Monthly Variance: -3,668
Most Over-Budget Category: Operatinal Costs
Most Under-Budget Category: Development Costs


In [ ]:
from prophet import Prophet



forecasts = {}

for col in cost_cols:
    train = actual[['Month', col]].copy()
    train.columns = ['ds', 'y']
    
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.1
    )
    m.fit(train)
    
    future = m.make_future_dataframe(periods=12, freq='MS')
    fc = m.predict(future)
    
    future_fc = fc[fc['ds'] > actual['Month'].max()][['ds','yhat','yhat_lower','yhat_upper']]
    forecasts[col] = future_fc.reset_index(drop=True)

print("=== 2024 FORECAST — ALL CATEGORIES ===\n")
summary = pd.DataFrame({'Month': forecasts[cost_cols[0]]['ds']})
for col in cost_cols:
    summary[col] = forecasts[col]['yhat'].clip(lower=0).round(0).astype(int)

summary['Total_Forecast'] = summary[cost_cols].sum(axis=1)
print(summary[['Month','Total_Forecast']].to_string())
print(f"\nTotal 2024 Forecast: {summary['Total_Forecast'].sum():,.0f}")
print(f"Avg Monthly Forecast: {summary['Total_Forecast'].mean():,.0f}")

18:12:33 - cmdstanpy - INFO - Chain [1] start processing
18:12:47 - cmdstanpy - INFO - Chain [1] done processing
18:12:47 - cmdstanpy - INFO - Chain [1] start processing
18:13:16 - cmdstanpy - INFO - Chain [1] done processing
18:13:17 - cmdstanpy - INFO - Chain [1] start processing
18:13:46 - cmdstanpy - INFO - Chain [1] done processing
18:13:46 - cmdstanpy - INFO - Chain [1] start processing
18:13:56 - cmdstanpy - INFO - Chain [1] done processing
18:13:56 - cmdstanpy - INFO - Chain [1] start processing
18:14:03 - cmdstanpy - INFO - Chain [1] done processing
18:14:03 - cmdstanpy - INFO - Chain [1] start processing
